In [ ]:
import pandas as pd
from pathlib import Path

# Arquivos de entrada e saída
arquivo_parquet = Path("02_PENSE2024_MICRODADOS_FILTRADO.parquet")
arquivo_dicionario = Path("00_Dicionário_Microdados_PENSE2024.xlsx")

arquivo_saida = Path("03_PENSE2024_MICRODADOS_COM_LABELS.parquet")

# Lê a base de dados
df = pd.read_parquet(arquivo_parquet)

# Lê o dicionário de variáveis
dic = pd.read_excel(
    arquivo_dicionario,
    sheet_name="PeNSE2024"
)

# Remove espaços dos códigos e descrições
dic["Nome da Variável"] = (ess
    dic["Nome da Variável"]
    .astype(str)
    .str.strip()
)

dic["Descrição da Variável"] = (
    dic["Descrição da Variável"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Cria o dicionário: código da coluna -> descrição da pergunta
mapa_labels = {}

for _, linha in dic.iterrows():

    codigo = str(linha["Nome da Variável"]).strip()
    descricao = str(linha["Descrição da Variável"]).strip()

    # Ignora registros sem código ou sem descrição
    if not codigo or codigo == "nan" or not descricao or descricao == "nan":
        continue

    mapa_labels[codigo] = descricao

print(f"Total de variáveis no dicionário: {len(mapa_labels)}")

# ============================================================
# CRIA O LABEL COMBINANDO CÓDIGO ORIGINAL + DESCRIÇÃO
# ============================================================
#
# Exemplo:
# código original: Q001
# descrição: Qual é o seu sexo?
#
# novo label:
# Q001 - Qual é o seu sexo?
#
# IMPORTANTE:
# O código original NÃO é removido.
# Apenas o nome (label) da coluna é alterado para:
# código - descrição
#

novos_nomes = []

for coluna in df.columns:

    # Mantém exatamente o nome/código original da coluna
    codigo = str(coluna).strip()

    # Procura a descrição correspondente no dicionário
    descricao = mapa_labels.get(codigo, "")

    if descricao:
        novo_nome = f"{codigo} - {descricao}"
    else:
        # Se não houver descrição, mantém somente o código
        novo_nome = codigo

    novos_nomes.append(novo_nome)

# Verifica se houve algum nome duplicado
contagem = {}
nomes_finais = []

for nome in novos_nomes:

    contagem[nome] = contagem.get(nome, 0) + 1

    if contagem[nome] == 1:
        nomes_finais.append(nome)
    else:
        nomes_finais.append(f"{nome} ({contagem[nome]})")

# Aplica os novos labels às colunas
df.columns = nomes_finais

# ============================================================
# VERIFICAÇÃO
# ============================================================

print("Conversão dos labels concluída!")
print("-" * 60)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

print("\nLabels finais das variáveis:")
for nome in df.columns:
    print(nome)

# Grava o novo arquivo Parquet
df.to_parquet(
    arquivo_saida,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print(f"\nArquivo salvo: {arquivo_saida.resolve()}")


Total de variáveis no dicionário: 425
Renomeação concluída!
--------------------------------------------------
Linhas: 153,870
Colunas: 41

Novos nomes das variáveis:
código de regiões
código UF
indica se é município da capital
Qual é o seu sexo?
IDADE DO ESTUDANTE, CATEGORIAS
Qual é a sua cor ou raça?
Você mora com sua mãe?
Você mora com seu pai?
Na sua casa tem computador ou notebook?
Você tem acesso à internet em sua casa?
NOS ÚLTIMOS 7 DIAS, sem contar as aulas de educação física da escola, em quantos dias você praticou alguma atividade física?
Quantas horas por dia você costuma ficar sentado(a), assistindo televisão, jogando videogame, usando computador, celular, tablet ou fazendo outras atividades sentado(a)? (NÃO contar sábado, domingo, feriados ou o tempo sentado na escola)
Alguma vez na vida você tomou um copo ou uma dose de bebida alcoólica?
NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável sabia realmente o que você estava fazendo em seu tempo livre?
NOS ÚL

In [ ]:
# Lê a base de dados
df = pd.read_parquet('03_PENSE2024_MICRODADOS_COM_LABELS.parquet')

# Inspecao da base de dados

# Exibe as primeiras linhas da base
display(df.head())

,código de regiões,código UF,indica se é município da capital,Qual é o seu sexo?,"IDADE DO ESTUDANTE, CATEGORIAS",Qual é a sua cor ou raça?,Você mora com sua mãe?,Você mora com seu pai?,Na sua casa tem computador ou notebook?,Você tem acesso à internet em sua casa?,...,"NOS ÚLTIMOS 30 DIAS, em quantos dias você não foi à escola porque não se sentia seguro NA ESCOLA?","NOS ÚLTIMOS 12 MESES, quantas vezes você foi agredido(a) fisicamente por sua mãe, pai ou responsável?","NOS ÚLTIMOS 12 MESES, quantas vezes você foi agredido(a) fisicamente por OUTRA PESSOA que não seja sua mãe, pai ou responsável?","Alguma vez na vida alguém o(a) tocou, manipulou, beijou ou expôs partes do corpo contra a sua vontade?","Alguma vez na vida alguém ameaçou, intimidou ou obrigou a ter relações sexuais ou qualquer outro ato sexual contra a sua vontade?","NOS ÚLTIMOS 12 MESES, a escola desenvolveu quais dessas seguintes ações? - Prevenção de violências e acidentes","NOS ÚLTIMOS 12 MESES, a escola desenvolveu quais dessas seguintes ações? - Prevenção de práticas de bullying nas dependências da escola","NOS ÚLTIMOS 12 MESES, a escola contou com algum suporte psicológico para situações de bullying e violências ocorridas na escola para alunos, professores e funcionários?",Quais tipos de suporte a escola oferece? - A escola possui em seu quadro de funcionários profissional de saúde psicológica/mental,Quais tipos de suporte a escola oferece? - A escola conta com serviço terceirizado de profissional de saúde psicológica/mental
0,1,11,0,2.0,1.0,4.0,2.0,1.0,2.0,1.0,...,1.0,1.0,1.0,-1.0,-1.0,1.0,0.0,1.0,1.0,0.0
1,1,11,0,2.0,2.0,4.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,2.0,2.0,1.0,0.0,1.0,1.0,0.0
2,1,11,0,1.0,1.0,4.0,1.0,1.0,2.0,1.0,...,1.0,1.0,1.0,-1.0,-1.0,0.0,1.0,4.0,-1.0,-1.0
3,1,11,0,1.0,2.0,4.0,1.0,2.0,2.0,1.0,...,1.0,1.0,1.0,2.0,2.0,1.0,0.0,1.0,1.0,0.0
4,1,11,0,1.0,1.0,4.0,1.0,2.0,1.0,1.0,...,1.0,1.0,1.0,-1.0,-1.0,0.0,1.0,4.0,-1.0,-1.0


In [ ]:
# Exibe numero de linhas e colunas da database
df.shape

(153870, 41)

In [ ]:
# Exibe estatisticas descritivas das colunas da database

# Salva o resumo estatístico em uma nova variável
resumo_estatisticas = df.describe()

# Para poder copiar em formato de tabela
resumo_estatisticas.to_clipboard()

# Verifica os tipos de dados de cada coluna calculada no resumo
print(resumo_estatisticas.dtypes)


código de regiões                                                                                                                                                                                                                                float64
código UF                                                                                                                                                                                                                                        float64
indica se é município da capital                                                                                                                                                                                                                 float64
Qual é o seu sexo?                                                                                                                                                                                                                               float64
IDAD

In [ ]:
import pandas as pd
import numpy as np

# Configurações de exibição do Pandas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ============================================================
# 1. CARREGA O DICIONÁRIO DO PENSE 2024
# ============================================================

df_dict = pd.read_excel(
    '00_Dicionário_Microdados_PENSE2024.xlsx',
    engine='openpyxl'
)

# ============================================================
# 2. FUNÇÃO PARA NORMALIZAR TEXTOS
# ============================================================

def normalizar_texto(texto):
    return "".join(str(texto).split()).lower()

# ============================================================
# 3. CRIA O DICIONÁRIO DE CATEGORIAS
#
# Chave: descrição da pergunta
# Valor: {código da resposta: descrição da resposta}
# ============================================================

mapeamento_perguntas = {}

for _, row in df_dict.iterrows():

    pergunta_desc = str(row['Descrição da Variável']).strip()
    chave_normalizada = normalizar_texto(pergunta_desc)

    categorias_texto = str(row['Categoria'])

    if categorias_texto and categorias_texto != 'nan':

        linhas_categoria = categorias_texto.split('\n')
        sub_mapa = {}

        for linha in linhas_categoria:

            if ' - ' in linha:
                partes = linha.split(' - ', 1)
                cod = partes[0].strip()
                desc = partes[1].strip()

                # Guarda diferentes formatos possíveis do código
                sub_mapa[cod] = desc

                try:
                    sub_mapa[str(int(float(cod)))] = desc
                except (ValueError, TypeError):
                    pass

                try:
                    sub_mapa[str(float(cod))] = desc
                except (ValueError, TypeError):
                    pass

        mapeamento_perguntas[chave_normalizada] = sub_mapa

# ============================================================
# 4. CONVERTE OS VALORES DAS 41 COLUNAS EM DESCRIÇÕES
#
# IMPORTANTE:
# As colunas agora possuem o formato:
#
# código - descrição
#
# Exemplo:
# Q001 - Qual é o seu sexo?
#
# O dicionário, porém, possui apenas:
#
# Nome da Variável = Q001
# Descrição da Variável = Qual é o seu sexo?
#
# Por isso, primeiro separamos o código da descrição.
# ============================================================

linhas_tabela_final = []

for col in df.columns:

    # Label completo da coluna
    label_coluna = str(col).strip()

    # --------------------------------------------------------
    # Separa o código da descrição
    # --------------------------------------------------------
    # O split acontece somente no primeiro " - ", pois algumas
    # descrições também podem conter " - ".
    if ' - ' in label_coluna:
        codigo_coluna, pergunta_desc = label_coluna.split(' - ', 1)
        codigo_coluna = codigo_coluna.strip()
        pergunta_desc = pergunta_desc.strip()
    else:
        # Segurança para colunas sem o formato esperado
        codigo_coluna = label_coluna
        pergunta_desc = label_coluna

    # --------------------------------------------------------
    # Localiza as categorias usando a DESCRIÇÃO DA PERGUNTA
    # --------------------------------------------------------
    chave_busca = normalizar_texto(pergunta_desc)
    mapa_coluna = mapeamento_perguntas.get(chave_busca, {})

    # Distribuição dos valores da coluna
    distribuicao = df[col].value_counts(
        normalize=True,
        dropna=False
    ) * 100

    linhas_respostas_pergunta = []

    for valor, pct in distribuicao.items():

        # Trata valores ausentes antes da conversão para texto
        if pd.isna(valor):
            descricao_resposta = 'Valores Nulos / Em Branco (NaN)'
        else:
            val_str = str(valor).strip()

            # Corrige valores como 1.0 -> 1
            if val_str.endswith('.0'):
                val_str = val_str[:-2]

            descricao_resposta = mapa_coluna.get(
                val_str,
                'Significado não encontrado no dicionário'
            )

        # Mantém o código da resposta junto com sua descrição
        texto_linha_resposta = (
            f'[{valor}] {descricao_resposta}: {pct:.2f}%'
        )

        linhas_respostas_pergunta.append(texto_linha_resposta)

    # Junta todas as respostas da pergunta em um único bloco
    bloco_respostas_completo = '\n'.join(
        linhas_respostas_pergunta
    )

    # Mantém o label completo da coluna na tabela final
    linhas_tabela_final.append({
        'Pergunta / Variável': label_coluna,
        'Distribuição das Respostas': bloco_respostas_completo
    })

# ============================================================
# 5. CRIA A TABELA FINAL
# ============================================================

df_consolidado = pd.DataFrame(linhas_tabela_final)

# ============================================================
# 6. COPIA PARA O CLIPBOARD
# ============================================================

df_consolidado.to_clipboard(
    index=False,
    sep='\t'
)

print('PROCESSO CONCLUÍDO!')
print('-' * 60)
print(f'Perguntas processadas: {len(df_consolidado)}')
print()

# Verifica quantas colunas encontraram suas categorias no dicionário
colunas_com_mapa = 0
colunas_sem_mapa = []

for col in df.columns:

    label_coluna = str(col).strip()

    if ' - ' in label_coluna:
        _, pergunta_desc = label_coluna.split(' - ', 1)
        pergunta_desc = pergunta_desc.strip()
    else:
        pergunta_desc = label_coluna

    chave_busca = normalizar_texto(pergunta_desc)

    if chave_busca in mapeamento_perguntas:
        colunas_com_mapa += 1
    else:
        colunas_sem_mapa.append(label_coluna)

print(f'Colunas com categorias encontradas: {colunas_com_mapa}')
print(f'Colunas sem categorias encontradas: {len(colunas_sem_mapa)}')

if colunas_sem_mapa:
    print('\nColunas sem correspondência no dicionário:')
    for nome in colunas_sem_mapa:
        print(nome)

print('\nTabela pronta para copiar para Excel/Pages.')

# Exibe todas as perguntas
display(
    df_consolidado.style
    .set_properties(**{
        'white-space': 'pre-wrap',
        'text-align': 'left',
        'vertical-align': 'top'
    })
)


✨ PROCESSO CONCLUÍDO! Tabela compacta de 2 colunas salva na memória.
Agora vá ao Pages (célula A1) ou Excel e aperte Cmd+V.


,Pergunta / Variável,Distribuição das Respostas
0,código de regiões,[2] nordeste: 33.29% [1] norte: 24.69% [3] sudeste: 18.56% [5] centro-oeste: 12.11% [4] sul: 11.35%
1,código UF,[32] Espírito Santo: 4.87% [35] São Paulo: 4.81% [33] Rio de Janeiro: 4.62% [26] Pernambuco: 4.59% [29] Bahia: 4.52% [11] Rondônia: 4.32% [31] Minas Gerais: 4.26% [41] Paraná: 4.11% [42] Santa Catarina: 4.04% [17] Tocantins: 4.04% [13] Amazonas: 3.95% [50] Mato Grosso do Sul: 3.92% [21] Maranhão: 3.85% [28] Sergipe: 3.83% [27] Alagoas: 3.82% [15] Pará: 3.51% [25] Paraíba: 3.44% [22] Piauí: 3.31% [12] Acre: 3.23% [43] Rio Grande do Sul: 3.20% [16] Amapá: 3.10% [23] Ceará: 2.97% [24] Rio Grande do Norte: 2.95% [51] Mato Grosso: 2.94% [52] Goiás: 2.84% [14] Roraima: 2.54% [53] Distrito Federal: 2.42%
2,indica se é município da capital,[1] é capital: 51.41% [0] não é capital: 48.59%
3,Qual é o seu sexo?,[2.0] Mulher: 48.38% [1.0] Homem: 48.28% [nan] Valores Nulos / Em Branco (NaN): 3.34%
4,"IDADE DO ESTUDANTE, CATEGORIAS",[2.0] 13 a 15 anos: 49.89% [3.0] 16 e 17 anos: 26.83% [1.0] Menos de 13 anos: 15.50% [4.0] 18 anos ou mais: 4.37% [nan] Valores Nulos / Em Branco (NaN): 3.34% [-9.0] Sem resposta: 0.08%
5,Qual é a sua cor ou raça?,[4.0] Parda: 41.79% [1.0] Branca: 39.18% [2.0] Preta: 10.31% [nan] Valores Nulos / Em Branco (NaN): 3.34% [3.0] Amarela: 2.66% [5.0] Indígena: 2.17% [-9.0] Sem resposta: 0.55% [-2.0] Abandono: 0.01%
6,Você mora com sua mãe?,[1.0] Sim: 86.93% [2.0] Não: 9.69% [nan] Valores Nulos / Em Branco (NaN): 3.34% [-9.0] Sem resposta: 0.03% [-2.0] Abandono: 0.01%
7,Você mora com seu pai?,[1.0] Sim: 61.33% [2.0] Não: 35.25% [nan] Valores Nulos / Em Branco (NaN): 3.34% [-9.0] Sem resposta: 0.06% [-2.0] Abandono: 0.01%
8,Na sua casa tem computador ou notebook?,[1.0] Sim: 61.15% [2.0] Não: 35.45% [nan] Valores Nulos / Em Branco (NaN): 3.34% [-9.0] Sem resposta: 0.04% [-2.0] Abandono: 0.02%
9,Você tem acesso à internet em sua casa?,[1.0] Sim: 94.01% [nan] Valores Nulos / Em Branco (NaN): 3.34% [2.0] Não: 2.60% [-9.0] Sem resposta: 0.03% [-2.0] Abandono: 0.02%
